In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
'''In the All of Us juypter notebook (Querying_conditions_concepts), write SQL code that selects patients with EHR data and genomics data and creates a pandas dataframe with the following columns, and save to file
concept_name
Concept_id
Patient count per concept
'''

In [ ]:

def get_viral_condition_concepts():
    """
    Returns one row per condition_concept_id (including all descendants of the passed-in concept_id)
    containing:
      - condition_concept_id
      - standard_concept_name
      - patient_count (number of distinct patients with that concept)
    Filters to patients with EHR + any genomics data, only “flat” events from the CB search table,
    and only standard concepts.
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      -- only people with both EHR & genomics
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant    = 1
            OR has_lr_whole_genome_variant = 1
            OR has_array_data              = 1
          )
      ),

      -- the “flat” events that CB searched over
      flat_events AS (
        SELECT DISTINCT person_id, concept_id
        FROM `{dataset}.cb_search_all_events`
      ),

      -- all standard‐concept descendants of your root concept_id
      target_concepts AS (
        SELECT DISTINCT descendant_concept_id AS condition_concept_id
        FROM `{dataset}.concept_ancestor`
        JOIN `{dataset}.concept` c
          ON descendant_concept_id = c.concept_id
        WHERE ancestor_concept_id = 440029
          AND c.standard_concept = 'S'
      )

    SELECT
      co.condition_concept_id,
      c_std.concept_name        AS standard_concept_name,
      COUNT(DISTINCT co.person_id) AS patient_count
    FROM `{dataset}.condition_occurrence` co

    -- only our filtered patients
    JOIN ehr_genomics_patients eg
      ON co.person_id = eg.person_id

    -- only events that show up in the CB flat‐events table
    JOIN flat_events fe
      ON co.person_id     = fe.person_id
     AND co.condition_concept_id = fe.concept_id

    -- restrict to the root concept + its descendants
    JOIN target_concepts tc
      ON co.condition_concept_id = tc.condition_concept_id

    -- grab the human‐readable name
    JOIN `{dataset}.concept` c_std
      ON co.condition_concept_id = c_std.concept_id

    GROUP BY
      co.condition_concept_id,
      standard_concept_name
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )

    # ensure the columns come back in the exact order you want
    return df[[
        "condition_concept_id",
        "standard_concept_name",
        "patient_count"
    ]]


In [ ]:
final_cohort_query = get_viral_condition_concepts()

In [ ]:
final_cohort_query

In [ ]:
cohorts = final_cohort_query[final_cohort_query['patient_count'] > 100]

In [ ]:
cohorts = cohorts.sort_values(by=['patient_count'], ascending= False)

In [ ]:
cohorts

In [ ]:
cohorts.to_csv("updated_concept_query_list_greater_than_100.csv")